Code for Introductory basic data visualization abilities in Python. 

Largely inspired from methods in Jessica Garwood's OC 301 Oceanographic Data Analysis class

Written by Drew Moreland

In [1]:
import cartopy #mapping
import cartopy.crs as ccrs #mapping
import cartopy.feature as cfeature #mapping
import numpy as np #lots of data steps
import scipy as sp #computing with data variables
import pandas as pd #data import and more
import xarray as xr #for importing .nc regional data
import cartopy.mpl.ticker as cticker #mapping
import matplotlib.colors as mcolors #color options
import matplotlib.pyplot as plt #plotting

#fun color options
import cmocean
oxy = cmocean.cm.oxy
solar = cmocean.cm.solar
thermal = cmocean.cm.thermal
algae = cmocean.cm.algae

Convert Data using Excel MET -->CSV

Practice good data naming practices intuitive for both you and others who may use the files



In [2]:
#Import Data as .csv

#use consistent naming structure (e.g. YYMMMDD.csv)

Aug25_26 = pd.read_csv("", skiprows=3, low_memory=False)

Aug26_26 = pd.read_csv("", skiprows=3, low_memory=False)

Aug27_26 = pd.read_csv("", skiprows=3, low_memory=False) 

TypeError: read_csv() missing 1 required positional argument: 'filepath_or_buffer'

In [ ]:
#processing NAs registered as various forms of -99
Aug25_26[np.isclose(Aug25_26, -99.0, atol=1e-4)] = np.nan
Aug26_26[np.isclose(Aug26_26, -99.0, atol=1e-4)] = np.nan
Aug27_26[np.isclose(Aug27_26, -99.0, atol=1e-4)] = np.nan

Mapping!

1. Import basemap


In [4]:
#importing basemap
####ADD IN GEBCO BATHYMETRY DATA
import rasterio
import os

path = os.path.expanduser("~/classes/OC396X/OC396X_Summer_2026/materials/gebco_imagery.tif")

src = rasterio.open(path)

img = src.read(1)
bounds = src.bounds

In [ ]:
#Setup framework for subplots for each year

ax1 = plt.subplot(1,3,1, projection=ccrs.PlateCarree()) #1 row, 3 columns, 1st graph (reading left to right)
ax2 = plt.subplot(1,3,2, projection=ccrs.PlateCarree()) #1 row, 3 columns, 2nd graph (reading left to right) 
ax3 = plt.subplot(1,3,3, projection=ccrs.PlateCarree()) #1 row, 3 columns, 3rd graph (reading left to right)


#specify plotting extent
x0 = -117   #western extent
x1 = -119.5  #eastern extent
y0 = 32.5   #southern extent
y1 = 33.8 #northern extent

#add basemap for each of the subplot
for ax in [ax1, ax2, ax3]:
    ax.set_extent([x0, x1, y0, y1])

    ax.imshow(
        img,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        transform=ccrs.PlateCarree(),
        cmap='gray',
        zorder=0,
        alpha=0.6 )#specify intensity of basemap color
    

#add data from each day to a subplot
norm = mcolors.Normalize(vmin=12, vmax=20) #specify min/max of color scheme
PC = ax1.scatter(Aug25_26.LO, Aug25_26.LA, c=Aug25_26.ST, cmap="gray", norm=norm, transform=ccrs.PlateCarree()) 
PC = ax2.scatter(Aug26_26.LO, Aug26_26.LA, c=Aug26_26.ST, cmap="gray", norm=norm, transform=ccrs.PlateCarree()) 
PC = ax3.scatter(Aug27_26.LO, Aug27_26.LA, c=Aug27_26.ST, cmap="gray", norm=norm, transform=ccrs.PlateCarree()) 



# Add and label colorbar
cb = fig.colorbar(PC, ax=ax7)
cb.set_label("Surface Temperature (C)")

# ticks 
for ax in [ax1, ax2, ax3]:
    ax.set_yticks(np.arange(32.5,33.79,0.25), crs=ccrs.PlateCarree()) #UPDATE
    ax.set_xticks(np.arange(-119,-117,0.5), crs=ccrs.PlateCarree()) #UPDATE

lat_formatter = cticker.LatitudeFormatter()
lon_formatter = cticker.LongitudeFormatter()

for ax in [ax1, ax2, ax3]:
    ax.yaxis.set_major_formatter(lat_formatter)
    ax.xaxis.set_major_formatter(lon_formatter)

# Add axes labels
for ax in [ax1, ax2, ax3]:
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)



#titles
ax1.set_title('August 25th')
ax2.set_title('August 26th')
ax3.set_title('August 27th')


In [ ]:
#Wind direction during cruise

fig= plt.figure(figsize=(8, 5))
ax1 = plt.subplot(1,3,1, projection=ccrs.PlateCarree())
ax2 = plt.subplot(1,3,2, projection=ccrs.PlateCarree())
ax3 = plt.subplot(1,3,3, projection=ccrs.PlateCarree())


#plotting extent
x0 = -117   
x1 = -119.5 
y0 = 32.5   
y1 = 33.8

#basemap
for ax in [ax1, ax2, ax3]:
    ax.set_extent([x0, x1, y0, y1])

    ax.imshow(
        img,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        transform=ccrs.PlateCarree(),
        cmap='gray',
        zorder=0,
        alpha=0.6
    )

# Convert wind speed & direction to U/V (horizontal/vertical) components

rad25 = np.deg2rad(Aug25_26.TI)
rad26 = np.deg2rad(Aug26_26.TI)
rad27 = np.deg2rad(Aug27_26.TI)

# Meteorological convention (wind direction = coming FROM)
U25 = -Aug25_26.TW * np.sin(rad25)# east-west component
U26 = -Aug26_26.TW * np.sin(rad26)
U27 = -Aug27_26.TW * np.sin(rad27)


V25_l1 = -Aug25_26.TW  * np.cos(rad25)  # north-south component
V26_l2 = -Aug26_26.TW * np.cos(rad26)
V27_l1 = -Aug27_26.TW * np.cos(rad27)


# -----------------------------
# Plot
# -----------------------------

# Scatter plot of spatial locations
#plt.scatter(lon, lat, color='black', s=20, label='Locations')

PC = ax1.scatter(Aug25_26.LO, Aug25_26.LA, color='gray', linewidth=0.5, alpha=0.5, zorder=1) 
PC = ax2.scatter(Aug26_26.LO, Aug26_26.LA, color='gray', linewidth=0.5, alpha=0.5, zorder=1) 
PC = ax3.scatter(Aug27_26.LO, Aug27_26.LA, color='gray', linewidth=0.5, alpha=0.5, zorder=1) 

# Quiver plot for wind vectors
norm = mcolors.Normalize(vmin=0, vmax=20)
q = ax1.quiver(Aug25_26.LO[::10], Aug25_26.LA[::10], U25_l1[::10], V22_l1[::10], leg1_22_clean.TW[::10], norm = norm, cmap='Spectral_r', scale=150)
q = ax2.quiver(Aug26_26.LO[::10], Aug26_26.LA[::10], U26_l2[::10], V22_l2[::10], leg2_22_clean.TW[::10], norm = norm, cmap='Spectral_r', scale=150)
q = ax3.quiver(Aug27_26.LO[::10], Aug27_26.LA[::10], U27_l1[::10], V23_l1[::10], leg1_23_clean.TW[::10], norm = norm, cmap='Spectral_r', scale=150)


cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) 

# Create the colorbar in that new axis
cb = fig.colorbar(q, cax=cbar_ax)
cb.set_label("Wind Intensity (m/s)", fontsize=12, fontweight='bold')
# ticks 
ax1.set_yticks(np.arange(32.5,33.8,0.5), crs=ccrs.PlateCarree())

for ax in [ax1, ax2, ax3]:
    ax.set_xticks(np.arange(-119.5,-117,1), crs=ccrs.PlateCarree())

lat_formatter = cticker.LatitudeFormatter()
lon_formatter = cticker.LongitudeFormatter()

for ax in  [ax1, ax2, ax3]:
    ax.yaxis.set_major_formatter(lat_formatter)
    ax.xaxis.set_major_formatter(lon_formatter)

# Add axes labels
for ax in  [ax1, ax2, ax3]:
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
#titles
ax1.set_title('Aug 25, 2026')
ax2.set_title('Aug 26, 2026')
ax3.set_title('Aug 27, 2026')


#fig.tight_layout(rect=[0, 0, 0.9, 1]) # rect leaves room for the colorbar on the right
#plt.show()

Basic graphing options

In [ ]:
fig= plt.figure(figsize=(8, 5))
ax1 = plt.subplot(1,3,1)
ax2 = plt.subplot(1,3,2)
ax3 = plt.subplot(1,3,3)


#histogram
ax1.hist(data.var1, data.var2)

#scatterplot
ax2.scatter(data.var1, data.var2)

#wind rose
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from windrose import WindroseAxes
import numpy as np



# Create the wind rose plot
ax3 = WindroseAxes.from_ax()
ax3.bar(compile22.TI, compile22.TW, normed=True, opening=0.8, edgecolor='white')

# Add a legend for wind speed bins
ax3.set_legend(title = "Wind Speed (m/s) 2022")
plt.show()

Compare with Regional climate (3 month satellite data average)
https://www.psl.noaa.gov/mddb2/makePlot.html?variableID=156646


In [ ]:
#import regional data using xarray with nc files
SST_when_to_when = xr.open_dataset('~/classes/OC396X/OC396X_Summer_2026/materials/____.nc')

#(might be needed) collapse time dimension (or else python might get confused with many options for time plotting
SSTAVG_when_to_when = SST_when_to_when['sst'].mean(dim='time')



In [ ]:
#Plot average regional SST with surface underway track


fig= plt.figure(figsize=(8, 8))
 
ax1 = plt.subplot(1,1,1, projection=ccrs.PlateCarree())

#plotting extent
x0 = -117   
x1 = -120 
y0 = 31   
y1 = 34

ax1.set_extent([x0, x1, y0, y1])
################################
# # Plot SST in a latitude, longitude map using pcolormesh
norm = mcolors.Normalize(vmin=12, vmax=20)
PC = ax1.pcolormesh(SSTAVG_when_to_when.lon, SSTAVG_when_to_when.lat, SSTAVG_when_to_when, cmap="Spectral_r", vmin=12, vmax=20, transform=ccrs.PlateCarree())
PC = ax1.scatter(Aug25_26.LO, Aug25_26.LA, c=Aug25_26.ST, cmap="Spectral_r", norm=norm, transform=ccrs.PlateCarree()) 

# Add and label colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) 

# Create the colorbar in that new axis
cb = fig.colorbar(PC, cax=cbar_ax)
cb.set_label("Surface Temperature (C)", fontsize=12, fontweight='bold')
 # This is how you create a label for your colorbar.
# Add axes labels
ax1.set_ylabel("longitude")
ax1.set_title('2026 Feb-Mar')
ax1.coastlines()
ax1.add_feature(cfeature.BORDERS, linewidth=0.5)

 


